# CWD Oscar Project

## Notebook 02 — Enrich Movie Metadata

This notebook enriches the cleaned Best Picture nominee dataset with movie metadata from OMDb for use in **The Oscar Formula** and **The Oscar Curse** analyses.

### Input
- `data/processed/best_picture_nominees_clean.csv`

### Output
- `data/processed/best_picture_nominees_enriched.csv`

In [1]:
!git clone https://github.com/mclary-claritywithdata/cwd-oscar-formula.git
%cd /content/cwd-oscar-formula

Cloning into 'cwd-oscar-formula'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 25 (delta 5), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 24.73 KiB | 12.36 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/cwd-oscar-formula


In [2]:
import pandas as pd
import numpy as np
import requests
import time

from google.colab import userdata

In [3]:
OMDB_API_KEY = userdata.get("OMDB_API_KEY")

if not OMDB_API_KEY:
    raise ValueError("OMDB_API_KEY secret not found in Colab secrets")

print("OMDb key loaded successfully.")

OMDb key loaded successfully.


In [4]:
bp = pd.read_csv("data/processed/best_picture_nominees_clean.csv")

print(bp.shape)
bp.head()

(624, 8)


,year,film,producers,year_raw,ceremony_year_raw,film_release_year,ceremony_number,winner_flag
0,1927.0,Wings,"Paramount (Lucien Hubbard, Jesse L. Lasky, B. ...",1927/28 (1st),1927/28 (1st),1927.0,1.0,NaN
1,1927.0,7th Heaven,"Fox (William Fox, producer)",1927/28 (1st),1927/28 (1st),1927.0,1.0,NaN
2,1927.0,The Racket,"The Caddo Company (Howard Hughes, producer)",1927/28 (1st),1927/28 (1st),1927.0,1.0,NaN
3,1928.0,The Broadway Melody,Metro-Goldwyn-Mayer (Irving Thalberg & Lawrenc...,1928/29 (2nd) [a],1928/29 (2nd) [a],1928.0,2.0,NaN
4,1928.0,Alibi,"Feature Productions (Roland West, producer)",1928/29 (2nd) [a],1928/29 (2nd) [a],1928.0,2.0,NaN


In [5]:
bp["film_clean"] = (
    bp["film"]
    .astype(str)
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

display(bp[["film", "film_clean"]].head(10))

,film,film_clean
0,Wings,Wings
1,7th Heaven,7th Heaven
2,The Racket,The Racket
3,The Broadway Melody,The Broadway Melody
4,Alibi,Alibi
5,The Hollywood Revue,The Hollywood Revue
6,In Old Arizona,In Old Arizona
7,The Patriot,The Patriot
8,All Quiet on the Western Front,All Quiet on the Western Front
9,The Big House,The Big House


In [6]:
def get_omdb_movie(title, year=None, api_key=OMDB_API_KEY):
    base_url = "https://www.omdbapi.com/"

    params = {
        "t": title,
        "apikey": api_key
    }

    if pd.notna(year):
        try:
            params["y"] = int(year)
        except Exception:
            pass

    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"Response": "False", "Error": str(e)}

In [7]:
test_result = get_omdb_movie("Oppenheimer", 2023)
test_result

{'Title': 'Oppenheimer',
 'Year': '2023',
 'Rated': 'R',
 'Released': '21 Jul 2023',
 'Runtime': '180 min',
 'Genre': 'Biography, Drama, History',
 'Director': 'Christopher Nolan',
 'Writer': 'Christopher Nolan, Kai Bird, Martin Sherwin',
 'Actors': 'Cillian Murphy, Emily Blunt, Matt Damon',
 'Plot': 'A dramatization of the life story of J. Robert Oppenheimer, the physicist who had a large hand in the development of the atomic bombs that brought an end to World War II.',
 'Language': 'English, German, Italian, Dutch',
 'Country': 'United States, United Kingdom',
 'Awards': 'Won 7 Oscars. 364 wins & 376 nominations total',
 'Poster': 'https://m.media-amazon.com/images/M/MV5BN2JkMDc5MGQtZjg3YS00NmFiLWIyZmQtZTJmNTM5MjVmYTQ4XkEyXkFqcGc@._V1_SX300.jpg',
 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.2/10'},
  {'Source': 'Rotten Tomatoes', 'Value': '93%'},
  {'Source': 'Metacritic', 'Value': '90/100'}],
 'Metascore': '90',
 'imdbRating': '8.2',
 'imdbVotes': '1,008,589',
 'im

In [8]:
movie_meta = []

for _, row in bp.iterrows():
    title = row["film_clean"]

    if "film_release_year" in bp.columns:
        year = row["film_release_year"]
    elif "year" in bp.columns:
        year = row["year"]
    else:
        year = None

    result = get_omdb_movie(title, year)

    movie_meta.append({
        "film": row["film"],
        "film_clean": title,
        "lookup_year": year,
        "omdb_title": result.get("Title"),
        "omdb_year": result.get("Year"),
        "released": result.get("Released"),
        "runtime": result.get("Runtime"),
        "genre": result.get("Genre"),
        "director": result.get("Director"),
        "writer": result.get("Writer"),
        "actors": result.get("Actors"),
        "plot": result.get("Plot"),
        "language": result.get("Language"),
        "country": result.get("Country"),
        "awards": result.get("Awards"),
        "poster": result.get("Poster"),
        "metascore": result.get("Metascore"),
        "imdb_rating": result.get("imdbRating"),
        "imdb_votes": result.get("imdbVotes"),
        "box_office": result.get("BoxOffice"),
        "production": result.get("Production"),
        "response_flag": result.get("Response"),
        "error_msg": result.get("Error")
    })

    time.sleep(0.2)

In [9]:
meta_df = pd.DataFrame(movie_meta)

print("Metadata shape:", meta_df.shape)
display(meta_df.head())

Metadata shape: (624, 23)


,film,film_clean,lookup_year,omdb_title,omdb_year,released,runtime,genre,director,writer,...,country,awards,poster,metascore,imdb_rating,imdb_votes,box_office,production,response_flag,error_msg
0,Wings,Wings,1927.0,Wings,1927,05 Jan 1929,144 min,"Drama, Romance, War","William A. Wellman, Harry d'Abbadie d'Arrast","John Monk Saunders, Hope Loring, Louis D. Lighton",...,United States,Won 2 Oscars. 9 wins & 1 nomination total,https://m.media-amazon.com/images/M/MV5BYjNiNj...,78,7.5,"15,487",N/A,N/A,True,None
1,7th Heaven,7th Heaven,1927.0,7th Heaven,1927,30 Oct 1927,110 min,"Drama, Romance",Frank Borzage,"Austin Strong, Benjamin Glazer, Katherine Hill...",...,United States,Won 3 Oscars. 9 wins & 2 nominations total,https://m.media-amazon.com/images/M/MV5BMTk2Mj...,N/A,7.5,"4,348",N/A,N/A,True,None
2,The Racket,The Racket,1927.0,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,False,Movie not found!
3,The Broadway Melody,The Broadway Melody,1928.0,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,False,Movie not found!
4,Alibi,Alibi,1928.0,Hubby's Latest Alibi,1928,04 Nov 1928,N/A,"Short, Comedy",Phil Whitman,"Alfred M. Loewenthal, Jefferson Moffitt, Phil ...",...,United States,N/A,N/A,N/A,N/A,N/A,N/A,N/A,True,None


In [10]:
meta_df["response_flag"].value_counts(dropna=False)

,count
response_flag,
True,341
False,283


In [11]:
failed_lookups = meta_df[meta_df["response_flag"] != "True"].copy()

print("Failed lookups:", failed_lookups.shape[0])
display(failed_lookups[["film", "film_clean", "lookup_year", "error_msg"]].head(20))

Failed lookups: 283


,film,film_clean,lookup_year,error_msg
2,The Racket,The Racket,1927.0,Movie not found!
3,The Broadway Melody,The Broadway Melody,1928.0,Movie not found!
5,The Hollywood Revue,The Hollywood Revue,1928.0,Movie not found!
8,All Quiet on the Western Front,All Quiet on the Western Front,1929.0,Movie not found!
9,The Big House,The Big House,1929.0,Movie not found!
11,The Divorcee,The Divorcee,1929.0,Movie not found!
13,Cimarron,Cimarron,1930.0,Movie not found!
14,East Lynne,East Lynne,1930.0,Movie not found!
15,The Front Page,The Front Page,1930.0,Movie not found!
16,Skippy,Skippy,1930.0,Movie not found!


In [12]:
bp_enriched = bp.merge(
    meta_df.drop(columns=["film_clean"]),
    on="film",
    how="left"
)

print("Enriched shape:", bp_enriched.shape)
display(bp_enriched.head())

Enriched shape: (650, 30)


,year,film,producers,year_raw,ceremony_year_raw,film_release_year,ceremony_number,winner_flag,film_clean,lookup_year,...,country,awards,poster,metascore,imdb_rating,imdb_votes,box_office,production,response_flag,error_msg
0,1927.0,Wings,"Paramount (Lucien Hubbard, Jesse L. Lasky, B. ...",1927/28 (1st),1927/28 (1st),1927.0,1.0,NaN,Wings,1927.0,...,United States,Won 2 Oscars. 9 wins & 1 nomination total,https://m.media-amazon.com/images/M/MV5BYjNiNj...,78,7.5,"15,487",N/A,N/A,True,None
1,1927.0,7th Heaven,"Fox (William Fox, producer)",1927/28 (1st),1927/28 (1st),1927.0,1.0,NaN,7th Heaven,1927.0,...,United States,Won 3 Oscars. 9 wins & 2 nominations total,https://m.media-amazon.com/images/M/MV5BMTk2Mj...,N/A,7.5,"4,348",N/A,N/A,True,None
2,1927.0,The Racket,"The Caddo Company (Howard Hughes, producer)",1927/28 (1st),1927/28 (1st),1927.0,1.0,NaN,The Racket,1927.0,...,None,None,None,None,None,None,None,None,False,Movie not found!
3,1928.0,The Broadway Melody,Metro-Goldwyn-Mayer (Irving Thalberg & Lawrenc...,1928/29 (2nd) [a],1928/29 (2nd) [a],1928.0,2.0,NaN,The Broadway Melody,1928.0,...,None,None,None,None,None,None,None,None,False,Movie not found!
4,1928.0,Alibi,"Feature Productions (Roland West, producer)",1928/29 (2nd) [a],1928/29 (2nd) [a],1928.0,2.0,NaN,Alibi,1928.0,...,United States,N/A,N/A,N/A,N/A,N/A,N/A,N/A,True,None


In [13]:
bp_enriched["released_dt"] = pd.to_datetime(bp_enriched["released"], errors="coerce")
bp_enriched["release_month"] = bp_enriched["released_dt"].dt.month
bp_enriched["release_year_omdb"] = bp_enriched["released_dt"].dt.year

bp_enriched["runtime_min"] = (
    bp_enriched["runtime"]
    .astype(str)
    .str.extract(r"(\d+)")
)
bp_enriched["runtime_min"] = pd.to_numeric(bp_enriched["runtime_min"], errors="coerce")

bp_enriched["imdb_rating_num"] = pd.to_numeric(bp_enriched["imdb_rating"], errors="coerce")
bp_enriched["metascore_num"] = pd.to_numeric(bp_enriched["metascore"], errors="coerce")

bp_enriched["imdb_votes_num"] = (
    bp_enriched["imdb_votes"]
    .astype(str)
    .str.replace(",", "", regex=False)
)
bp_enriched["imdb_votes_num"] = pd.to_numeric(bp_enriched["imdb_votes_num"], errors="coerce")

bp_enriched["box_office_num"] = (
    bp_enriched["box_office"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
bp_enriched["box_office_num"] = pd.to_numeric(bp_enriched["box_office_num"], errors="coerce")

In [14]:
qa_cols = [
    "film",
    "released",
    "runtime",
    "runtime_min",
    "genre",
    "director",
    "imdb_rating",
    "imdb_rating_num",
    "box_office",
    "box_office_num",
    "response_flag",
    "error_msg"
]

display(bp_enriched[qa_cols].head(15))

,film,released,runtime,runtime_min,genre,director,imdb_rating,imdb_rating_num,box_office,box_office_num,response_flag,error_msg
0,Wings,05 Jan 1929,144 min,144.0,"Drama, Romance, War","William A. Wellman, Harry d'Abbadie d'Arrast",7.5,7.5,N/A,NaN,True,None
1,7th Heaven,30 Oct 1927,110 min,110.0,"Drama, Romance",Frank Borzage,7.5,7.5,N/A,NaN,True,None
2,The Racket,None,None,NaN,None,None,None,NaN,None,NaN,False,Movie not found!
3,The Broadway Melody,None,None,NaN,None,None,None,NaN,None,NaN,False,Movie not found!
4,Alibi,04 Nov 1928,N/A,NaN,"Short, Comedy",Phil Whitman,N/A,NaN,N/A,NaN,True,None
5,The Hollywood Revue,None,None,NaN,None,None,None,NaN,None,NaN,False,Movie not found!
6,In Old Arizona,20 Jan 1929,95 min,95.0,"Drama, Western","Irving Cummings, Raoul Walsh",5.5,5.5,"$2,834,000",2834000.0,True,None
7,The Patriot,01 Sep 1928,113 min,113.0,"Drama, History, Thriller",Ernst Lubitsch,N/A,NaN,N/A,NaN,True,None
8,All Quiet on the Western Front,None,None,NaN,None,None,None,NaN,None,NaN,False,Movie not found!
9,All Quiet on the Western Front,None,None,NaN,None,None,None,NaN,None,NaN,False,401 Client Error: Unauthorized for url: https:...


In [15]:
bp_enriched[[
    "released",
    "runtime_min",
    "genre",
    "director",
    "imdb_rating_num",
    "box_office_num"
]].isna().sum()

,0
released,298
runtime_min,299
genre,298
director,298
imdb_rating_num,301
box_office_num,491


In [16]:
bp_enriched.to_csv("data/processed/best_picture_nominees_enriched.csv", index=False)

print("Saved: data/processed/best_picture_nominees_enriched.csv")

Saved: data/processed/best_picture_nominees_enriched.csv


In [17]:
!ls data/processed

best_picture_nominees_clean.csv  best_picture_nominees_enriched.csv


In [18]:
total_rows = len(bp_enriched)
matched_rows = (bp_enriched["response_flag"] == "True").sum()
failed_rows = total_rows - matched_rows

print(f"Total rows: {total_rows}")
print(f"Matched rows: {matched_rows}")
print(f"Failed rows: {failed_rows}")
print(f"Match rate: {matched_rows / total_rows:.1%}")

Total rows: 650
Matched rows: 352
Failed rows: 298
Match rate: 54.2%
